NOTE: These results are stimulated as we don't have live data of user behaviour after intervetion.

# Business scenario question:

Can a targeted retention intervention can reduce the churn among those high risk customers?

From SHAP analysis we know contract month-to-month feature is prone to churn more. Goal is to identify which customers whould likely to change their behaviour because of retention intervention. 

# Intervention: 
Offer retention incentive to selected high risk customers

- personalized discount
- offer services with discounted price
- free security add-on trial
- contract upgrade incentive 
- loyalty reward

# Primary outcome

Customer churn after intervention

# Business Objectives

1. Did the intervention reduced churn?
2. Whats the avg treatment effect (ATE)?
3. which customer segment responded strongly?
4. Incremental revenue
5. ROI


In [12]:
#loading data
from pathlib import Path
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in (cwd, *cwd.parents) if (path / "code").is_dir() and (path / "model").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate the project root containing code/ and model/")

DATA_DIR = PROJECT_ROOT / "code" / "data"
MODEL_DIR = PROJECT_ROOT / "model"
df = pd.read_csv(DATA_DIR / "Edited_Data.csv")

In [8]:
df.shape
df.columns.tolist()

['gender',
 'SeniorCitizen',
 'Partner',
 'Dependents',
 'tenure',
 'PhoneService',
 'PaperlessBilling',
 'MonthlyCharges',
 'TotalCharges',
 'Churn',
 'avg_monthly_charge',
 'charge_ratio',
 'service_score',
 'MultipleLines_No',
 'MultipleLines_No phone service',
 'MultipleLines_Yes',
 'InternetService_DSL',
 'InternetService_Fiber optic',
 'InternetService_No',
 'OnlineSecurity_No',
 'OnlineSecurity_No internet service',
 'OnlineSecurity_Yes',
 'OnlineBackup_No',
 'OnlineBackup_No internet service',
 'OnlineBackup_Yes',
 'DeviceProtection_No',
 'DeviceProtection_No internet service',
 'DeviceProtection_Yes',
 'TechSupport_No',
 'TechSupport_No internet service',
 'TechSupport_Yes',
 'StreamingTV_No',
 'StreamingTV_No internet service',
 'StreamingTV_Yes',
 'StreamingMovies_No',
 'StreamingMovies_No internet service',
 'StreamingMovies_Yes',
 'Contract_Month-to-month',
 'Contract_One year',
 'Contract_Two year',
 'PaymentMethod_Bank transfer (automatic)',
 'PaymentMethod_Credit card (

In [14]:
best_xgb  = joblib.load(MODEL_DIR / "xgb_best_model.pkl")
scaler    = joblib.load(MODEL_DIR / "scaler.pkl")
feat_names = joblib.load(MODEL_DIR / "feature_names.pkl")


In [ ]:
causal_df = df.copy()
X = causal_df.loc[:, feat_names]
X_scaled_df = pd.DataFrame(scaler.transform(X), columns=feat_names)
causal_df['churn_probability'] = best_xgb.predict_proba(X_scaled_df)[:, 1]